# 2. How the agent searched for the fusion design

**What this notebook shows.** How the fusion design was found (TASK.md Step 2), from the saved
records only; nothing is trained here.

**The agent has two layers.**
1. **Inner agent (deterministic, no language model).** Inside one *version*, it runs at most 20
   experiments on seed 0's development patients. It opens one candidate per family, follows the
   family that leads, drops a family once it is clearly behind, tunes the best few, and picks the
   simplest candidate that the best one does not *clearly* beat (clearly = wins in ≥ 90% of 2,000
   paired bootstrap resamples). Every step is written to an append-only log with its reason.
2. **Outer layer (a language-model agent, between versions).** It reads the logs, writes down why
   the version did or did not work, and designs the next version with **one** new idea,
   pre-registered (hypothesis, prediction, what would prove it wrong) before the run.
   Its protocol is `docs/loop.md`; everything it decided is in `decisions/` (one file per version).

The test sets were never used during the search.

In [ ]:
# Run from anywhere inside the repository: move to its root so every path below works.
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from src.eval import final_view as fv

fv.style()
pd.set_option("display.max_colwidth", 200)
print("Repository:", root.name)

## 1. The 11 versions

- *Agent's winner*: what the inner agent's selection rule chose.
- *Best experiment*: the highest development AUROC in that version (often not chosen, because it
  did not *clearly* beat a simpler candidate).
- *Winner's curse*: how much choosing the best of many experiments inflates its score (estimated
  over all experiments so far); small values mean selection noise is under control.
- `_uvg` rows: two earlier versions re-run once the third model (GenBio-PathFM) was available.

In [ ]:
versions = fv.versions_table("artifacts/versions")
versions

In [ ]:
fv.plot_versions(versions); plt.show()

**What the versions taught.**
- **v04** brought the one big jump: reading individual tiles with learned attention instead of
  averaging all tiles (+0.055).
- Every later idea moved the score by less than 0.01: separate models per foundation model
  (v05), different tile pooling (v06–v08), ensembles (v09), age and sex choosing tiles (v10).
- **v11** (an extra coarse training target: grade groups) was the only later change that helped
  beyond its control; its winner H1 became the final design.
- Age and sex added 0 to +0.005 in every form tried.

The full reasoning, version by version: `decisions/SUMMARY.md` and `decisions/v01.md` … `v11.md`.

In [ ]:
display(Markdown(Path("decisions/SUMMARY.md").read_text()))

## 2. Inside one version: the decision log of v11

Every decision the inner agent took, with its rule and its reason, in order:

In [ ]:
fv.decisions("v11")

And every experiment it ran (development AUROC over 8 folds, gain over row 5):

In [ ]:
fv.experiments("v11")

The full human-readable log (each experiment, why it was chosen, all scores) is
`artifacts/versions/v11/seed_0/agent/decision_log.md`; the raw record is `trials.jsonl` next to it.
Change `"v11"` above to any other version (`"v01"` … `"v11"`) to see its decisions.

## 3. Reproduce a version

Each version is the shared code plus its config (`conf/versions/vNN.yaml`). To repeat one on the
development data (about 30–60 minutes on one GPU):

```bash
uv run --no-sync python -m scripts.10_fusion_search --seed 0 --dev --config conf/versions/v11.yaml
```

In this repository the run's record already exists, so the agent reads it, recognises every
experiment as done and adds nothing (resume). To repeat a version from scratch, run the command in
a fresh clone (with the cached features copied in).